# Notebook 04 – Reversible PII Masking

**Project:** IndiaPII – India-Specific PII Detection and Reversible Masking  
**Environment:** Local VS Code / Jupyter (Python 3.11)

This notebook loads the fine-tuned MuRIL token-classification model from the local project, detects PII spans, replaces them with typed placeholders, and restores the original text using an in-memory mapping.

> Before running: download/copy the final trained model from Google Drive into `models/muril-pii/`. This notebook does not train the model. The mapping used here is an in-memory demonstration; production storage must be access-controlled and encrypted.


In [5]:
# Cell 1 – Imports and configuration
from pathlib import Path
import re
from collections import defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models" / "muril-pii"

print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available() if hasattr(torch.backends, "mps") else False)

if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_DIR}. Copy the trained muril_final folder "
        "from Google Drive into models/muril-pii, or update MODEL_DIR."
    )


Project root: /Users/navyaa09/IndiaPII
Model directory: /Users/navyaa09/IndiaPII/models/muril-pii
CUDA available: False
MPS available: True


In [6]:
# Cell 2 – Load the fine-tuned MuRIL model
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR))

if torch.cuda.is_available():
    device = 0
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = -1

pii_pipeline = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

print("Model loaded.")
print("Device:", device)
print("Labels:", model.config.id2label)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6198.83it/s]


Model loaded.
Device: mps
Labels: {0: 'O', 1: 'B-PERSON', 2: 'I-PERSON', 3: 'B-ADDRESS', 4: 'I-ADDRESS', 5: 'B-PHONE', 6: 'I-PHONE', 7: 'B-AADHAAR', 8: 'I-AADHAAR', 9: 'B-PAN', 10: 'I-PAN', 11: 'B-EMAIL', 12: 'I-EMAIL', 13: 'B-UPI', 14: 'I-UPI'}


## 1. Detect PII entities

The model returns character offsets for detected entities. We retain these offsets so replacement happens against the original text, rather than reconstructing text from tokens.


In [15]:

# Cell 3 – Normalize labels and detect PII with exact character offsets

def normalize_entity_label(raw_label: str) -> str:
    label = raw_label.upper()
    label = re.sub(r"^[BILU]-", "", label)
    return label


def detect_pii(text: str, score_threshold: float = 0.0):
    predictions = pii_pipeline(text)
    entities = []

    for item in predictions:
        start = int(item["start"])
        end = int(item["end"])

        label = normalize_entity_label(
            item.get("entity_group", item.get("entity", ""))
        )

        score = float(item.get("score", 0.0))

        if start < 0 or end > len(text) or start >= end:
            continue

        if score < score_threshold:
            continue

        # PAN boundary correction:
        # Keep only the actual PAN number, excluding words such as "PAN".
        if label == "PAN":
            detected_text = text[start:end]
            match = re.search(
                r"[A-Z]{5}[0-9]{4}[A-Z]",
                detected_text
            )

            if match:
                original_start = start
                start = original_start + match.start()
                end = original_start + match.end()

        entities.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end],
            "score": score,
        })

    # Resolve overlapping predictions
    entities.sort(
        key=lambda x: (
            x["start"],
            -(x["end"] - x["start"]),
            -x["score"]
        )
    )

    selected = []

    for entity in entities:
        if selected and entity["start"] < selected[-1]["end"]:
            if entity["score"] > selected[-1]["score"]:
                selected[-1] = entity
            continue

        selected.append(entity)

    return selected


# Test detection
sample_text = (
    "My name is Rahul Sharma and my email is rahul@example.com."
)

detected = detect_pii(sample_text)

for entity in detected:
    print(entity)


{'start': 11, 'end': 23, 'label': 'PERSON', 'text': 'Rahul Sharma', 'score': 0.10297642648220062}
{'start': 40, 'end': 57, 'label': 'EMAIL', 'text': 'rahul@example.com', 'score': 0.1198437437415123}


## 2. Reversible masking

Mask from right to left so replacing a later span does not shift the offsets of earlier spans. Repeated occurrences of the same exact value and label reuse the same placeholder within one masking operation.


In [14]:
# Cell 4 – Mask text and create reversible mapping
def mask_text(text: str, entities=None):
    if entities is None:
        entities = detect_pii(text)

    # Validate and sort spans; reject overlaps rather than silently corrupting text.
    entities = sorted(entities, key=lambda x: (x["start"], x["end"]))
    for i, entity in enumerate(entities):
        if not (0 <= entity["start"] < entity["end"] <= len(text)):
            raise ValueError(f"Invalid entity offsets: {entity}")
        if i > 0 and entity["start"] < entities[i - 1]["end"]:
            raise ValueError("Overlapping entity spans detected. Resolve them before masking.")

    counters = defaultdict(int)
    value_to_placeholder = {}
    placeholder_to_value = {}
    replacements = []

    for entity in entities:
        label = normalize_entity_label(entity["label"])
        value = text[entity["start"]:entity["end"]]
        key = (label, value)

        if key not in value_to_placeholder:
            counters[label] += 1
            placeholder = f"[{label}_{counters[label]}]"
            value_to_placeholder[key] = placeholder
            placeholder_to_value[placeholder] = value
        else:
            placeholder = value_to_placeholder[key]

        replacements.append((entity["start"], entity["end"], placeholder))

    masked = text
    for start, end, placeholder in reversed(replacements):
        masked = masked[:start] + placeholder + masked[end:]

    return masked, placeholder_to_value

masked_text, pii_mapping = mask_text(sample_text, detected)
print("Original:", sample_text)
print("Masked:  ", masked_text)
print("Mapping: ", pii_mapping)


Original: My name is Rahul Sharma and my email is rahul@example.com.
Masked:   My name is [PERSON_1] and my email is [EMAIL_1].
Mapping:  {'[PERSON_1]': 'Rahul Sharma', '[EMAIL_1]': 'rahul@example.com'}


In [16]:
# Cell 5 – Restore original text
def unmask_text(masked_text: str, mapping: dict):
    restored = masked_text
    # Longer keys first in case placeholder names ever share prefixes.
    for placeholder in sorted(mapping, key=len, reverse=True):
        restored = restored.replace(placeholder, mapping[placeholder])
    return restored

restored_text = unmask_text(masked_text, pii_mapping)
print("Restored:", restored_text)
assert restored_text == sample_text
print("Round-trip restoration passed.")


Restored: My name is Rahul Sharma and my email is rahul@example.com.
Round-trip restoration passed.


## 3. Test multiple Indian-language examples

These are synthetic test strings. Inspect the detections before interpreting the output; model performance on synthetic examples does not establish real-world accuracy.


In [17]:
# Cell 6 – Multilingual smoke tests
test_texts = [
    "My name is Rahul Sharma and my email is rahul@example.com.",
    "मेरा नाम अमित शर्मा है और मेरा फोन नंबर 9876543210 है।",
    "Mera naam Priya hai, mera PAN ABCDE1234F hai.",
]

for text in test_texts:
    print("\\nTEXT:", text)
    entities = detect_pii(text)
    for entity in entities:
        print("  DETECTED:", entity["label"], repr(entity["text"]), entity["start"], entity["end"])
    masked, mapping = mask_text(text, entities)
    restored = unmask_text(masked, mapping)
    print("  MASKED:  ", masked)
    print("  RESTORED:", restored)
    assert restored == text


\nTEXT: My name is Rahul Sharma and my email is rahul@example.com.
  DETECTED: PERSON 'Rahul Sharma' 11 23
  DETECTED: EMAIL 'rahul@example.com' 40 57
  MASKED:   My name is [PERSON_1] and my email is [EMAIL_1].
  RESTORED: My name is Rahul Sharma and my email is rahul@example.com.
\nTEXT: मेरा नाम अमित शर्मा है और मेरा फोन नंबर 9876543210 है।
  DETECTED: PERSON 'अमित शर्मा' 9 19
  DETECTED: PHONE '9876543210' 40 50
  MASKED:   मेरा नाम [PERSON_1] है और मेरा फोन नंबर [PHONE_1] है।
  RESTORED: मेरा नाम अमित शर्मा है और मेरा फोन नंबर 9876543210 है।
\nTEXT: Mera naam Priya hai, mera PAN ABCDE1234F hai.
  DETECTED: PERSON 'Priya' 10 15
  DETECTED: PAN 'ABCDE1234F' 30 40
  MASKED:   Mera naam [PERSON_1] hai, mera PAN [PAN_1] hai.
  RESTORED: Mera naam Priya hai, mera PAN ABCDE1234F hai.


## 4. Edge-case checks

The following checks validate the masking logic independently of model quality: repeated values, adjacent entities, Unicode text, and exact restoration.


In [18]:
# Cell 7 – Unit tests with manually specified spans
def run_masking_unit_tests():
    # Repeated exact value should reuse the same placeholder.
    text = "Amit called Amit."
    entities = [
        {"start": 0, "end": 4, "label": "PERSON", "score": 1.0},
        {"start": 12, "end": 16, "label": "PERSON", "score": 1.0},
    ]
    masked, mapping = mask_text(text, entities)
    assert masked == "[PERSON_1] called [PERSON_1]."
    assert unmask_text(masked, mapping) == text

    # Unicode offsets must restore correctly.
    text = "नमस्ते अमित"
    start = text.index("अमित")
    entities = [{"start": start, "end": start + len("अमित"), "label": "PERSON", "score": 1.0}]
    masked, mapping = mask_text(text, entities)
    assert unmask_text(masked, mapping) == text

    # Adjacent spans should both be replaced.
    text = "Amit9876543210"
    entities = [
        {"start": 0, "end": 4, "label": "PERSON", "score": 1.0},
        {"start": 4, "end": 14, "label": "PHONE", "score": 1.0},
    ]
    masked, mapping = mask_text(text, entities)
    assert masked == "[PERSON_1][PHONE_1]"
    assert unmask_text(masked, mapping) == text

    # Overlapping spans must fail safely.
    try:
        mask_text("abcdef", [
            {"start": 0, "end": 4, "label": "PERSON", "score": 0.9},
            {"start": 2, "end": 5, "label": "ADDRESS", "score": 0.8},
        ])
    except ValueError:
        pass
    else:
        raise AssertionError("Overlapping spans should raise ValueError.")

    print("All masking unit tests passed.")

run_masking_unit_tests()


All masking unit tests passed.


## 5. Save a small demonstration report

Only masked text and aggregate counts are saved by default. Do not save the original-to-placeholder mapping in plaintext for real user data.


In [19]:
# Cell 8 – Save demonstration output (masked text only)
from datetime import datetime
import json

RESULTS_DIR = PROJECT_ROOT / "results" / "masking"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

demo_record = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "original_text_used_for_demo": sample_text,
    "masked_text": masked_text,
    "entity_count": len(detected),
    "entity_labels": [entity["label"] for entity in detected],
    "round_trip_passed": restored_text == sample_text,
}

# This is a synthetic demonstration record. For real PII, omit original_text_used_for_demo.
output_path = RESULTS_DIR / "masking_demo.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(demo_record, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)


Saved: /Users/navyaa09/IndiaPII/results/masking/masking_demo.json


## Completion checklist

- [ ] Fine-tuned model loads from `models/muril-pii/`
- [ ] English, Hindi and Hinglish smoke tests run
- [ ] Masking replaces detected spans without changing surrounding text
- [ ] Unmasking restores the exact original text
- [ ] Repeated values reuse placeholders as intended
- [ ] Unicode and adjacent-span tests pass
- [ ] Overlapping spans fail safely

**Security note:** This notebook demonstrates reversible masking, not production-grade secure vaulting. For deployment, keep the mapping server-side, isolate it per request/session, encrypt it at rest, restrict access, and avoid logging original PII or mappings.
